In [ ]:
%pip install -q langgraph langchain-openai langchain-core python-dotenv grandalf

# 01 · `create_react_agent` + tool (prebuilt)

Prebuilt agent with 2 automatic nodes:
- **`agent`** — LLM with bound tools. If it emits a `tool_call` → goes to tools.
- **`tools`** — executes the tool and returns the result to the agent.

```
user → [agent] → tool_call? → [tools] → back to [agent] → final response
```

The loop repeats until the LLM responds without a `tool_call` → `END`.

In [2]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

## Custom tool

In [4]:
@tool
def get_weather(city: str) -> str:
    """Returns the current weather for a given city."""
    return f"Sunny, 22 °C in {city}"

## Create the prebuilt agent

In [5]:
llm = ChatOpenAI(model="gpt-4o-mini")
graph = create_react_agent(llm, tools=[get_weather])

/var/folders/fn/1rdtxjl13c72x6x62ddyxcdm0000gn/T/ipykernel_16261/2497645337.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph = create_react_agent(llm, tools=[get_weather])


## Visualize the graph (2 nodes)

In [6]:
graph.get_graph().print_ascii()

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +-------+           
          | agent |           
          +-------+*          
          .         *         
        ..           **       
       .               *      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


## Invoke the agent

In [ ]:
result = graph.invoke({"messages": [{"role": "user", "content": "Weather in Madrid?"}]})

# Show all intermediate messages to see the full cycle
for msg in result["messages"]:
    print(f"{msg.__class__.__name__}: {msg.content}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"  tool_calls: {msg.tool_calls}")

## Final response

In [8]:
print(result["messages"][-1].content)

The weather in Madrid is sunny with a temperature of 22 °C.
